### setup 

use analy env

what script does
- copy yaml file from /n/home07/than157/desktop/done-large_projects/learn-better/evolm/pretrain/lit-trainer/config_hub/custom_configs/pretrain/model.yaml
- save new yaml file at /n/home07/than157/desktop/done-large_projects/learn-better/evolm/pretrain/lit-trainer/config_hub/custom_configs/val/model-val.yaml
- make changes to new yaml file

In [1]:
import shutil
from itertools import product
import re


In [2]:
model_name_lst = [
    #0.5B
    'llama-0.5B-10BT-weightdecay0.0001-seed42',
    'llama-0.5B-10BT-weightdecay0.001-seed42',
    'llama-0.5B-10BT-weightdecay0.01-seed42',
    'llama-0.5B-10BT-weightdecay0.1-seed42', ###ALREADY DONE
    'llama-0.5B-10BT-weightdecay0.5-seed42',
    'llama-0.5B-10BT-weightdecay1.0-seed42',
    'llama-0.5B-10BT-weightdecay1.5-seed42',
    'llama-0.5B-10BT-weightdecay3.0-seed42',
    'llama-0.5B-10BT-weightdecay10.0-seed42',
    #1B
    'llama-1B-20BT-weightdecay0.0001-seed42',
    'llama-1B-20BT-weightdecay0.001-seed42',
    'llama-1B-20BT-weightdecay0.01-seed42',
    'llama-1B-20BT-weightdecay0.1-seed42',
    'llama-1B-20BT-weightdecay0.5-seed42',
    'llama-1B-20BT-weightdecay1.0-seed42',
    'llama-1B-20BT-weightdecay1.5-seed42',
    'llama-1B-20BT-weightdecay3.0-seed42',
    'llama-1B-20BT-weightdecay10.0-seed42',
    #4B
    'llama-4B-80BT-weightdecay1.0-seed42',
]

In [ ]:
n_newconfig_files_created = 0

for model_name in model_name_lst:
    print(f"Creating config file for: {model_name}")

    #make a copy of the ref yaml file
    og_config_file_path = f'lit-trainer/config_hub/custom_configs/pretrain/{model_name}.yaml'
    new_config_file_path = f'lit-trainer/config_hub/custom_configs/val/{model_name}-val.yaml'
    shutil.copy(og_config_file_path, new_config_file_path)

    ### make edits to the new config file
    with open(new_config_file_path, "r", encoding="utf-8") as f:
        text = f.read()
        
        #change out_dir
        old_str = f'out_dir: models/pretrained/{model_name}'
        new_str = f'out_dir: /n/netscratch/doshi-velez_lab/Everyone/models/val/{model_name}-val'
        text = text.replace(old_str, new_str)

        #change resume
        old_str = 'resume: false'
        new_str = f'resume: models/pretrained/{model_name}/final/lit_model.pth'
        text = text.replace(old_str, new_str)

        #change max_tokens
        match = re.search(r"(max_tokens:\s*\S+)", text) #extract "max_tokens: X"
        old_str = match.group(1)
        new_str = 'max_tokens: 0'
        text = text.replace(old_str, new_str)

        #change initial_validation (under eval)
        old_str = 'initial_validation: false'
        new_str = 'initial_validation: true'
        text = text.replace(old_str, new_str)

        #change max_iters (under eval)
        old_str = 'max_iters: 100'
        new_str = 'max_iters: 1526'
        text = text.replace(old_str, new_str)
        
        #change num_nodes
        #num_nodes is already 1 for 0.5B models
        #num_nodes=2 and num_nodes=4 for 1B + 4B models 
        match = re.search(r"(num_nodes:\s*\S+)", text) #extract "num_nodes: X"
        old_str = match.group(1)
        new_str = 'num_nodes: 1'
        text = text.replace(old_str, new_str)

        #change logger_run_id -- no longer matters since no longer using wandb (see below)
        old_str = 'logger_run_id: ' + 'llama2-' + '-'.join(model_name.split('-')[1:])
        new_str = old_str + '-val'
        text = text.replace(old_str, new_str)

        #change logger_name -- ULTIMATELY DECIDED TO TURN OFF WANDB LOGGING, use log files to see printed loss
        old_str = 'logger_name: wandb'
        new_str = 'logger_name: csv'
        text = text.replace(old_str, new_str)

        #write new config file
        with open(new_config_file_path, "w", encoding="utf-8") as f:
            f.write(text)

    n_newconfig_files_created += 1

print(n_newconfig_files_created)
print("Complete!")

Creating config file for: llama-0.5B-10BT-weightdecay0.0001-seed42
Creating config file for: llama-0.5B-10BT-weightdecay0.001-seed42
Creating config file for: llama-0.5B-10BT-weightdecay0.01-seed42
Creating config file for: llama-0.5B-10BT-weightdecay0.1-seed42
Creating config file for: llama-0.5B-10BT-weightdecay0.5-seed42
Creating config file for: llama-0.5B-10BT-weightdecay1.0-seed42
Creating config file for: llama-0.5B-10BT-weightdecay1.5-seed42
Creating config file for: llama-0.5B-10BT-weightdecay3.0-seed42
Creating config file for: llama-0.5B-10BT-weightdecay10.0-seed42
Creating config file for: llama-1B-20BT-weightdecay0.0001-seed42
Creating config file for: llama-1B-20BT-weightdecay0.001-seed42
Creating config file for: llama-1B-20BT-weightdecay0.01-seed42
Creating config file for: llama-1B-20BT-weightdecay0.1-seed42
Creating config file for: llama-1B-20BT-weightdecay0.5-seed42
Creating config file for: llama-1B-20BT-weightdecay1.0-seed42
Creating config file for: llama-1B-20BT

In [8]:
old_str

'logger_name: wandb'

In [7]:
print(text)


# The name of the model to pretrain. Choose from names in ``litgpt.config``. Mutually exclusive with
# ``model_config``. (type: Optional[str], default: null)
model_name: myllama-500M

# A ``litgpt.Config`` object to define the model architecture. Mutually exclusive with
# ``model_config``. (type: Optional[Config], default: null)
model_config:

# Directory in which to save checkpoints and logs. If running in a Lightning Studio Job, look for it in
# /teamspace/jobs/<job-name>/share. (type: <class 'Path'>, default: out/pretrain)
out_dir: models/val/llama-0.5B-10BT-weightdecay0.001-seed42-val

# The precision to use for pretraining. Possible choices: "bf16-true", "bf16-mixed", "32-true". (type: Optional[str], default: null)
precision: bf16-mixed

# Optional path to a checkpoint directory to initialize the model from.
# Useful for continued pretraining. Mutually exclusive with ``resume``. (type: Optional[Path], default: null)
initial_checkpoint_dir:

# Path to a checkpoint directory to res

In [ ]:
# out_dir: models/pretrained/pythia-14m --> models/compute_val_loss/pythia-14m
# resume: models/pretrained/pythia-14m/final/lit_model.pth
# max_tokens: X --> max_tokens: 0
# eval.final_validation: false --> final_validation: true
# eval.max_iters: 100 --> max_iters: 611 -- for 20M validation tokens
    # tokens_per_batch = (micro_batch_size * max_seq_length) = 16 * 2048 = 32768
    # n_batches = n_tokens_val / tokens_per_batch = 20000641 / 32768 = 611
    # NOTE: max_iters = n_batches
# logger_run_id: pythia-14m --> pythia-14m-val-loss



# 4B manually change micro_batch_size: 4 --> micro_batch_size: 16

#keep as is
# initial_checkpoint_dir: [empty] --> KEEP EMPTY /n/home07/than157/desktop/done-large_projects/learn-better/evolm/pretrain/lit-trainer/models/pretrained/pythia-14m/final
# resume: false --> resume: [empty] --> ACUTLLY CHANGE
# data: FineWeb -- should alread by fineweb for llama models